A Neuron from Scratch

Build a single neuron step by step, no libraries needed




In [2]:
# ===========================================
# A SINGLE NEURON - The building block of AI
# ===========================================

def neuron(inputs, weights, bias, activation='relu'):
  """
    One neuron = one tiny decision maker

    inputs:  What the neuron sees [x1, x2, x3, ...]
    weights: How important each input is [w1, w2, w3, ...]
    bias:    The neuron's "default mood" (shifts threshold)
  """
  # STEP 1: Multiply each input by its weight
  # "How much does each piece of evidence matter?"
  weighted = [x * w for x,w in zip(inputs, weights)]
  print(f"Weighted inputs: {[round(w,2) for w in weighted]}")

  # STEP 2: Sum everything + bias
  # "What's my total score?"
  total = sum(weighted) + bias
  print(f"Sum + bias: {round(total,2)}")

  # STEP 3: Apply activation
  # "What's my final decision?"
  if activation == 'relu':
    output = max(0,total)
  elif activation == 'sigmoid':
    import math
    total = max(-500, min(500,total)) # clipped to prevent overflow
    output = 1 / (1 + math.exp(-total))
  else:
    output = total # linear (no activation)

  print(f"Output ({activation}): {round(output,2)}")
  return output

# ===========================================
# TRY IT: Is this email spam?
# ===========================================
# Inputs: [has_urgent, has_link, from_known_sender, has_unsubscribe]

inputs = [1, 1, 0 , 1] # 1 yes, 0 no

# Weights: learned from data
# Positive = suggests spam, Negative = suggests not spam
weights = [0.8,   # "urgent" suggests spam
           0.5,   # links suggest spam
          -0.9,   # known sender suggests NOT spam
          -0.7]   # unsubscribe suggests NOT spam (legit emails have it)

bias = -0.3 # slightly skeptical by default
print("Is this email spam?")
print(f"Inputs:  {inputs}")
print(f"Weights: {weights}")
print(f"Bias:    {bias}")
print()
result = neuron(inputs, weights, bias, activation='sigmoid')
print(f"\nSpam probability (sigmoid): {round(result * 100)}%")


result = neuron(inputs, weights, bias)
print(f"\nSpam probability (relu): {round(result * 100)}%")




Is this email spam?
Inputs:  [1, 1, 0, 1]
Weights: [0.8, 0.5, -0.9, -0.7]
Bias:    -0.3

Weighted inputs: [0.8, 0.5, -0.0, -0.7]
Sum + bias: 0.3
Output (sigmoid): 0.57

Spam probability (sigmoid): 57%
Weighted inputs: [0.8, 0.5, -0.0, -0.7]
Sum + bias: 0.3
Output (relu): 0.3

Spam probability (relu): 30%


From Neuron to Network (PyTorch)

Stack neurons into layers, layers into a digit classifier




In [3]:
import torch
import torch.nn as nn

# ===========================================
# BUILDING A NEURAL NETWORK
# ===========================================

# A "Dense Layer" = many neurons in parallel
# nn.Linear(in, out) creates a layer with:
# - 'in' inputs going to 'out' neurons
# - Each neuron has 'in' weights + 1 bias

# MNIST digit classifier: 28x28 image → digit 0-9

model = nn.Sequential(
    # layer 1 : 784 inputs -> 128 neurons
    nn.Linear(784,128), nn.ReLU(),

    # Layer 2: 128 → 64 neurons
    nn.Linear(128, 64),
    nn.ReLU(),

    # Output: 64 → 10 (one score per digit)
    nn.Linear(64, 10)
    # No activation here - we'll use softmax for probabilities
)



# ===========================================
# HOW MANY PARAMETERS?
# ===========================================

print(f"Parameter count per layer")
print("-" * 40)
total = 0

for name, param in model.named_parameters():
  n = param.numel()
  total += n
  print(f"{name:15} {str(list(param.shape)):15}  = {n:,}")

print("-" * 40)
print(f"{'TOTAL':15} {'':15} = {total:,}")

# ===========================================
# FORWARD PASS: Input → Output
# ===========================================
print("\n Forward Pass example:")
fake_image = torch.randn(1,784)
output = model(fake_image)

print(f"Input shape:  {list(fake_image.shape)}")  # [1, 784]
print(f"Output shape: {list(output.shape)}")      # [1, 10]
print(f"Output scores: {[round(x, 2) for x in output[0].tolist()]}")
print(f"Predicted digit: {output.argmax().item()}")


# ===========================================
# DEBUG TIP: Print shapes at each layer
# ===========================================
print("\nShape at each layer:")
x = fake_image
for i, layer in enumerate(model):
    x = layer(x)
    print(f"  After layer {i} ({layer.__class__.__name__:10}): {list(x.shape)}")

Parameter count per layer
----------------------------------------
0.weight        [128, 784]       = 100,352
0.bias          [128]            = 128
2.weight        [64, 128]        = 8,192
2.bias          [64]             = 64
4.weight        [10, 64]         = 640
4.bias          [10]             = 10
----------------------------------------
TOTAL                           = 109,386

 Forward Pass example:
Input shape:  [1, 784]
Output shape: [1, 10]
Output scores: [0.17, 0.16, -0.1, 0.0, -0.01, -0.04, -0.05, 0.02, 0.15, 0.08]
Predicted digit: 0

Shape at each layer:
  After layer 0 (Linear    ): [1, 128]
  After layer 1 (ReLU      ): [1, 128]
  After layer 2 (Linear    ): [1, 64]
  After layer 3 (ReLU      ): [1, 64]
  After layer 4 (Linear    ): [1, 10]
